In [6]:
import pandas as pd
import requests
import re
from bs4 import BeautifulSoup

def scrape_delhi_metro_wiki():
    url = "https://en.wikipedia.org/wiki/List_of_Delhi_Metro_stations"
    print(f"Fetching data from {url}...")
    
    # Add a User-Agent so Wikipedia thinks we are a normal web browser
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }
    
    # Pass the headers into the request
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        print(f"Failed to fetch page. Status code: {response.status_code}")
        return

    # Parse with BeautifulSoup to ensure we capture the table structure properly
    soup = BeautifulSoup(response.content, 'html.parser')
    
    # Read HTML tables using pandas
    tables = pd.read_html(str(soup))
    
    # Find the main station table 
    # The correct table will have columns containing 'Station' and 'Line'
    df_stations = None
    for table in tables:
        # Flatten MultiIndex columns if Wikipedia uses grouped headers
        if isinstance(table.columns, pd.MultiIndex):
            table.columns = ['_'.join(col).strip() for col in table.columns.values]
            
        col_names = [str(c).lower() for c in table.columns]
        if any('station' in c for c in col_names) and any('line' in c for c in col_names):
            df_stations = table
            break

    if df_stations is None:
        print("Error: Could not locate the station table on the Wikipedia page.")
        return

    print("Table found. Cleaning data...")

    # Data Cleaning Function
    def clean_text(text):
        if pd.isna(text):
            return text
        text = str(text)
        # Remove Wikipedia reference brackets like [1], [a], [note 1]
        text = re.sub(r'\[.*?\]', '', text)
        # Remove dagger symbol (†) used for notes
        text = text.replace('†', '')
        # Remove multiple whitespaces/newlines
        text = re.sub(r'\s+', ' ', text)
        return text.strip()

    # Apply cleaning to all string-based columns
    for col in df_stations.columns:
        if df_stations[col].dtype == 'object':
            df_stations[col] = df_stations[col].apply(clean_text)

    # Rename columns to be more standard and easy to join with the Kaggle dataset
    # We will rename based on standard Wikipedia headers for this page, ignoring exact casing
    rename_mapping = {}
    for col in df_stations.columns:
        lower_col = col.lower()
        if 'station' in lower_col: rename_mapping[col] = 'Station_Name'
        elif 'line' in lower_col: rename_mapping[col] = 'Metro_Line'
        elif 'opened' in lower_col: rename_mapping[col] = 'Opening_Date'
        elif 'layout' in lower_col: rename_mapping[col] = 'Station_Layout'
        elif 'interchange' in lower_col or 'connection' in lower_col: rename_mapping[col] = 'Interchange_Connections'

    df_stations.rename(columns=rename_mapping, inplace=True)

    # Export to CSV
    output_filename = "data/wiki_delhi_metro_stations.csv"
    df_stations.to_csv(output_filename, index=False)
    
    print(f"\nSuccess! Extracted {len(df_stations)} stations.")
    print(f"Dataset saved as '{output_filename}'")
    
    # Preview the first few rows
    print("\nDataset Preview:")
    print(df_stations[['Station_Name', 'Metro_Line', 'Opening_Date']].head())

if __name__ == "__main__":
    scrape_delhi_metro_wiki()

Fetching data from https://en.wikipedia.org/wiki/List_of_Delhi_Metro_stations...
Table found. Cleaning data...

Success! Extracted 271 stations.
Dataset saved as 'wiki_delhi_metro_stations.csv'

Dataset Preview:
    Station_Name Station_Name   Metro_Line      Opening_Date
0   Adarsh Nagar     Elevated  Yellow Line   4 February 2009
1          AIIMS  Underground  Yellow Line  3 September 2010
2     Akshardham     Elevated    Blue Line  12 November 2009
3  Anand Vihar**     Elevated    Blue Line    6 January 2010
4  Anand Vihar**     Elevated    Pink Line   31 October 2018


/var/folders/gc/k740v3612cv_1m5s2d93wxzr0000gn/T/ipykernel_65972/3477220563.py:25: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(str(soup))
